# DA6401 Report - Section 2.5 Dead Neuron Investigation

Controlled high-learning-rate comparison (`lr=0.1`) between:
- ReLU
- Tanh

Includes dead-neuron counting, activation histograms, and convergence/gradient comparison plots.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
import wandb

import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

print("Project root:", ROOT)


In [ ]:
from ann.neural_network import NeuralNetwork
from utils.data_loader import load_dataset

RUN_EXPERIMENT = False  # Set to True to launch runs.

PROJECT = "da6401_assignment_1"
ENTITY = None
RUN_GROUP = "report_v1_2_5_dead_neuron"

common = dict(
    dataset="mnist",
    epochs=10,
    batch_size=64,
    loss="cross_entropy",
    optimizer="rmsprop",
    learning_rate=0.1,
    weight_decay=0.0,
    num_layers=3,
    hidden_size=[128, 128, 128],
    weight_init="xavier",
    wandb_project=PROJECT,
    wandb_entity=ENTITY,
    wandb_mode="online",
    seed=42,
)

configs = [
    {"activation": "relu", "label": "relu_lr0p1"},
    {"activation": "tanh", "label": "tanh_lr0p1"},
]

out_dir = ROOT / "src" / "tmp"
out_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
def activation_analysis(model, X, batch_size=256):
    num_layers = len(model.hidden_sizes)
    dead_counts = [np.zeros(h, dtype=np.int64) for h in model.hidden_sizes]
    total_seen = np.zeros(num_layers, dtype=np.int64)
    pooled = [[] for _ in range(num_layers)]

    for start in range(0, X.shape[0], batch_size):
        end = min(start + batch_size, X.shape[0])
        _ = model.forward(X[start:end])
        for li, act in enumerate(model.hidden_activations):
            dead_counts[li] += np.sum(act == 0.0, axis=0)
            total_seen[li] += act.shape[0]
            flat = act.ravel()
            if flat.size > 20000:
                idx = np.random.default_rng(42).choice(flat.size, size=20000, replace=False)
                flat = flat[idx]
            pooled[li].append(flat)

    out = []
    for li in range(num_layers):
        zeros_per_neuron = dead_counts[li]
        samples = int(total_seen[li])
        dead_mask = zeros_per_neuron == samples
        dead_neurons = int(np.sum(dead_mask))
        dead_ratio = dead_neurons / float(len(dead_mask))
        pooled_vals = np.concatenate(pooled[li]) if pooled[li] else np.array([], dtype=np.float64)
        out.append({
            "layer": li,
            "neurons": int(len(dead_mask)),
            "samples_seen": samples,
            "dead_neurons": dead_neurons,
            "dead_ratio": dead_ratio,
            "activation_values": pooled_vals,
        })
    return out


In [ ]:
results = []

if RUN_EXPERIMENT:
    data = load_dataset(dataset="mnist", seed=42)

    for cfg in configs:
        args = SimpleNamespace(
            activation=cfg["activation"],
            model_path=str(out_dir / f"model_2_5_{cfg['label']}.npy"),
            config_path=str(out_dir / f"config_2_5_{cfg['label']}.json"),
            **common,
        )

        run = wandb.init(
            project=PROJECT,
            entity=ENTITY,
            name=f"report_2_5_{cfg['label']}",
            group=RUN_GROUP,
            tags=["report_v1", "report_section_2.5", "dead_neuron", "mnist", cfg["activation"], "lr_0.1"],
            config=vars(args),
        )

        model = NeuralNetwork(args)
        history = model.train(
            X_train=data["X_train"],
            y_train=data["y_train"],
            epochs=args.epochs,
            batch_size=args.batch_size,
            X_val=data["X_val"],
            y_val=data["y_val"],
            X_test=data["X_test"],
            y_test=data["y_test"],
            wandb_run=run,
        )

        final_val = model.evaluate(data["X_val"], data["y_val"], batch_size=args.batch_size)
        final_test = model.evaluate(data["X_test"], data["y_test"], batch_size=args.batch_size)
        layer_stats = activation_analysis(model, data["X_val"], batch_size=256)

        dead_table = wandb.Table(columns=["layer", "neurons", "samples_seen", "dead_neurons", "dead_ratio"])
        for ls in layer_stats:
            dead_table.add_data(ls["layer"], ls["neurons"], ls["samples_seen"], ls["dead_neurons"], ls["dead_ratio"])
            run.log({f"analysis/activation_hist_l{ls['layer']}": wandb.Histogram(ls["activation_values"])})

        run.log({"analysis/dead_neuron_table": dead_table})

        val_acc_curve = [float(ep["val"]["accuracy"]) for ep in history]
        grad_curve = [float(ep["grad_norm_first_layer_mean"]) for ep in history]
        dead_frac_curve = [float(ep["dead_neuron_fraction_layer1_mean"]) for ep in history]

        rec = {
            "label": cfg["label"],
            "activation": cfg["activation"],
            "run_id": run.id,
            "run_name": run.name,
            "val_acc_curve": val_acc_curve,
            "grad_norm_curve": grad_curve,
            "dead_frac_curve": dead_frac_curve,
            "layer_stats": [
                {"layer": ls["layer"], "neurons": ls["neurons"], "dead_neurons": ls["dead_neurons"], "dead_ratio": ls["dead_ratio"]}
                for ls in layer_stats
            ],
            "final_val_acc": float(final_val["accuracy"]),
            "final_val_f1": float(final_val["f1"]),
            "final_test_acc": float(final_test["accuracy"]),
            "final_test_f1": float(final_test["f1"]),
        }
        results.append(rec)

        run.summary["analysis/final_val_acc"] = rec["final_val_acc"]
        run.summary["analysis/final_test_acc"] = rec["final_test_acc"]
        run.summary["analysis/dead_neurons_layer0"] = rec["layer_stats"][0]["dead_neurons"]
        run.summary["analysis/dead_ratio_layer0"] = rec["layer_stats"][0]["dead_ratio"]
        run.summary["analysis/early_plateau_val_acc_std_epochs_3_10"] = float(np.std(val_acc_curve[2:]))
        run.finish()

    plt.figure(figsize=(9, 5))
    for rec in results:
        plt.plot(range(1, len(rec["val_acc_curve"]) + 1), rec["val_acc_curve"], marker="o", linewidth=2, label=rec["label"])
    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy")
    plt.title("2.5 Validation Accuracy (LR=0.1, RMSProp, 3x128)")
    plt.grid(alpha=0.3)
    plt.legend()
    val_plot_path = out_dir / "report_2_5_val_accuracy_comparison.png"
    plt.tight_layout()
    plt.savefig(val_plot_path, dpi=180)
    plt.close()

    plt.figure(figsize=(9, 5))
    for rec in results:
        plt.plot(range(1, len(rec["grad_norm_curve"]) + 1), rec["grad_norm_curve"], marker="o", linewidth=2, label=rec["label"])
    plt.xlabel("Epoch")
    plt.ylabel("Grad Norm (First Hidden Layer)")
    plt.title("2.5 Gradient Norm Comparison (LR=0.1)")
    plt.grid(alpha=0.3)
    plt.legend()
    grad_plot_path = out_dir / "report_2_5_gradnorm_comparison.png"
    plt.tight_layout()
    plt.savefig(grad_plot_path, dpi=180)
    plt.close()

    summary_run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name="report_2_5_summary",
        group=RUN_GROUP,
        tags=["report_v1", "report_section_2.5", "dead_neuron", "summary"],
    )

    table = wandb.Table(columns=[
        "label", "activation", "run_id", "final_val_acc", "final_val_f1", "final_test_acc", "final_test_f1",
        "dead_l0", "dead_ratio_l0", "dead_l1", "dead_ratio_l1", "dead_l2", "dead_ratio_l2"
    ])
    for rec in results:
        ls = rec["layer_stats"]
        table.add_data(
            rec["label"], rec["activation"], rec["run_id"],
            rec["final_val_acc"], rec["final_val_f1"], rec["final_test_acc"], rec["final_test_f1"],
            ls[0]["dead_neurons"], ls[0]["dead_ratio"],
            ls[1]["dead_neurons"], ls[1]["dead_ratio"],
            ls[2]["dead_neurons"], ls[2]["dead_ratio"],
        )

    summary_run.log({
        "analysis/dead_neuron_comparison_table": table,
        "analysis/val_accuracy_comparison_plot": wandb.Image(str(val_plot_path)),
        "analysis/gradnorm_comparison_plot": wandb.Image(str(grad_plot_path)),
    })
    summary_run.finish()

    payload = {
        "setup": {
            "dataset": "mnist", "optimizer": "rmsprop", "learning_rate": 0.1,
            "loss": "cross_entropy", "batch_size": 64, "epochs": 10,
            "architecture": "3x128",
        },
        "results": results,
        "summary_run_name": "report_2_5_summary",
        "plots": {"val_accuracy": str(val_plot_path), "gradnorm": str(grad_plot_path)},
    }
    with open(out_dir / "report_2_5_dead_neuron.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print("Completed 2.5 runs and summary")
else:
    print("RUN_EXPERIMENT is False. No runs were launched.")


In [ ]:
summary_path = out_dir / "report_2_5_dead_neuron.json"
if summary_path.exists():
    data = json.loads(summary_path.read_text())
    for rec in data["results"]:
        ls = rec["layer_stats"]
        print(rec["label"], rec["run_id"], "val_acc", round(rec["final_val_acc"], 6), "dead", [x["dead_neurons"] for x in ls])
else:
    print("No summary file yet. Run experiment cells first.")
